In [ ]:
# Bloco R0 — baixar os arquivos oficiais da RSNA e conferir o conteúdo (passo 3.2)
import os, json, zipfile, time
import requests
import pandas as pd

base = "https://s3.amazonaws.com/east1.public.rsna.org/AI/2018/"
arquivos = {
    "mapeamento":  "pneumonia-challenge-dataset-mappings_2018.json",
    "anotacoes":   "pneumonia-challenge-annotations-adjudicated-kaggle_2018.json",
    "anot_inicial": "pneumonia-challenge-annotations-original_2018.json",
    "imagens":     "pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip",
}
os.makedirs("/out/rsna", exist_ok=True)

def baixar(nome_arq):
    destino = f"/out/rsna/{nome_arq}"
    if os.path.exists(destino):
        return destino
    t0 = time.time()
    with requests.get(base + nome_arq, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloco in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(bloco)
    print(f"  baixado em {(time.time() - t0) / 60:.1f} min")
    return destino

for chave, nome_arq in arquivos.items():
    print(f"{chave}: {nome_arq}")
    caminho = baixar(nome_arq)
    print(f"  tamanho: {os.path.getsize(caminho) / 1e9:.2f} GB")

# conteúdo do pacote de imagens, sem descompactar
with zipfile.ZipFile("/out/rsna/" + arquivos["imagens"]) as z:
    nomes = [i for i in z.infolist() if not i.is_dir()]
    ext = pd.Series([os.path.splitext(i.filename)[1].lower() for i in nomes]).value_counts()
    print("\nArquivos no pacote:", len(nomes))
    print("Extensões:", ext.to_dict())
    print("Tamanho descompactado:", round(sum(i.file_size for i in nomes) / 1e9, 2), "GB")
    print("Exemplos:", [i.filename for i in nomes[:3]])

# primeira olhada nos JSON
for chave in ["mapeamento", "anotacoes"]:
    with open("/out/rsna/" + arquivos[chave]) as f:
        dado = json.load(f)
    print(f"\n=== {chave}: tipo {type(dado).__name__}, tamanho {len(dado)}")
    exemplo = dado[0] if isinstance(dado, list) else {k: dado[k] for k in list(dado)[:2]}
    print(json.dumps(exemplo, indent=1)[:1200])

In [ ]:
# Bloco R1 — estrutura das anotações (MD.ai) e cobertura das 30 mil imagens
anot = json.load(open("/out/rsna/" + arquivos["anotacoes"]))
mapa = pd.DataFrame(json.load(open("/out/rsna/" + arquivos["mapeamento"])))

print("Chaves:", list(anot.keys()))
for k, v in anot.items():
    if isinstance(v, list):
        print(f"  {k}: lista com {len(v)} itens")

# definições das classes
nomes = {l["id"]: l["name"] for g in anot.get("labelGroups", []) for l in g.get("labels", [])}
print("\nClasses definidas:", nomes)

# todas as anotações, de todos os conjuntos do projeto MD.ai
todas = pd.concat([pd.DataFrame(ds.get("annotations", [])).assign(conjunto=ds.get("name"))
                   for ds in anot.get("datasets", [])], ignore_index=True)
print("\nColunas das anotações:", list(todas.columns))
todas["rotulo"] = todas["labelId"].map(nomes)
print("\nImagens por conjunto e classe:")
print(todas.groupby(["conjunto", "rotulo"])["SOPInstanceUID"].nunique().to_string())

# classe por imagem: opacidade se houver pelo menos uma caixa de opacidade
por_img = todas.groupby("SOPInstanceUID")["rotulo"].apply(
    lambda s: "Lung Opacity" if s.eq("Lung Opacity").any() else ";".join(sorted(set(s.dropna()))))
mapa = mapa.merge(por_img.rename("classe_img"), left_on="SOPInstanceUID", right_index=True, how="left")
mapa["classe_img"] = mapa["classe_img"].fillna("SEM ANOTACAO")

print("\nClasse por imagem (30 mil):")
print(mapa["classe_img"].value_counts().to_string())
print("\nsubset_group × classe:")
print(pd.crosstab(mapa["subset_group"], mapa["classe_img"]).to_string())

# paciente pelo nome original do NIH
mapa["nih_paciente"] = mapa["img_id"].str[:8]
ipp = mapa.groupby("nih_paciente").size()
print(f"\nPacientes NIH: {mapa['nih_paciente'].nunique()} | imagens por paciente: mediana {ipp.median():.0f}, máx {ipp.max()}")

mapa.to_csv("/out/rsna/rsna_mapa_rotulos.csv", index=False)
print("Salvo: /out/rsna/rsna_mapa_rotulos.csv")

In [ ]:
# Bloco R2 — rótulo final adjudicado por imagem
print("Grupos de rótulos:")
for g in anot["labelGroups"]:
    print(f"  {g.get('name')}: {[l['name'].strip() for l in g.get('labels', [])]}")

grupo_final = next(g for g in anot["labelGroups"]
                   if any(l["name"].strip() == "Lung Opacity" for l in g["labels"]))
ids_final = {l["id"]: l["name"].strip() for l in grupo_final["labels"]}
print("\nGrupo final:", grupo_final.get("name"), ids_final)

fin_ = todas[todas["labelId"].isin(ids_final)].copy()
fin_["classe"] = fin_["labelId"].map(ids_final)
comb = fin_.groupby("SOPInstanceUID")["classe"].agg(lambda s: " + ".join(sorted(set(s))))
print("\nCombinações de classe final por imagem:")
print(comb.value_counts().to_string())

def classe_final(c):
    for p in ["Lung Opacity", "No Lung Opacity / Not Normal", "Normal"]:
        if p in c:
            return p
    return "outro"

mapa["classe_final"] = mapa["SOPInstanceUID"].map(comb.map(classe_final)).fillna("SEM CLASSE FINAL")
ids_excl = [i for i, n in nomes.items() if n.strip() == "Exclude"]
mapa["exclude"] = mapa["SOPInstanceUID"].isin(set(todas.loc[todas["labelId"].isin(ids_excl), "SOPInstanceUID"]))

print("\nClasse final (30 mil):")
print(mapa["classe_final"].value_counts().to_string())
print("\nClasse final × marcada Exclude:")
print(pd.crosstab(mapa["classe_final"], mapa["exclude"]).to_string())
print("\nsubset_group × classe final:")
print(pd.crosstab(mapa["subset_group"], mapa["classe_final"]).to_string())

caixas = fin_[fin_["classe"] == "Lung Opacity"].groupby("SOPInstanceUID").size()
print("\nCaixas de opacidade por imagem positiva:")
print(caixas.describe().round(2).to_string())

mapa.drop(columns=["classe_img"]).to_csv("/out/rsna/rsna_mapa_rotulos.csv", index=False)
print("\nSalvo: /out/rsna/rsna_mapa_rotulos.csv")

In [ ]:
# Bloco R2b — correção: classe final por igualdade exata
mapa["classe_final"] = mapa["SOPInstanceUID"].map(comb).fillna("SEM CLASSE FINAL")
mapa.loc[mapa["exclude"] & mapa["classe_final"].eq("SEM CLASSE FINAL"), "classe_final"] = "EXCLUDE"
mapa["positivo"] = mapa["classe_final"].eq("Lung Opacity")

print("Classe final (30 mil):")
print(mapa["classe_final"].value_counts().to_string())

rot = mapa[mapa["classe_final"].isin(["Lung Opacity", "No Lung Opacity / Not Normal", "Normal"])]
print(f"\nCom rótulo final: {len(rot)} | positivos: {rot['positivo'].sum()} ({rot['positivo'].mean():.1%}) "
      f"| pacientes: {rot['nih_paciente'].nunique()}")
print("Pacientes com pelo menos 1 positivo:", rot.groupby("nih_paciente")["positivo"].any().sum())

mapa.drop(columns=["classe_img"], errors="ignore").to_csv("/out/rsna/rsna_mapa_rotulos.csv", index=False)
print("Salvo: /out/rsna/rsna_mapa_rotulos.csv")

In [ ]:
# Bloco R3c — retomar o envio até completar (versão independente, até 30 rodadas)
import os, glob, time, zipfile
import redivis

zip_rsna = "/out/rsna/pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip"
todos = glob.glob("/scratch/rsna_dcm/**/*.dcm", recursive=True)
if len(todos) < 30000:
    print("Arquivos locais incompletos; descompactando de novo...")
    os.makedirs("/scratch/rsna_dcm", exist_ok=True)
    with zipfile.ZipFile(zip_rsna) as z:
        z.extractall("/scratch/rsna_dcm")
    todos = glob.glob("/scratch/rsna_dcm/**/*.dcm", recursive=True)
print("Arquivos locais:", len(todos))

tab = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018:2t9m", version="next").table("dicom:sjtd")

def ja_enviados():
    return {getattr(f, "name", None) or f.properties.get("name") for f in tab.list_files()}

for rodada in range(1, 30):
    ja = ja_enviados()
    faltam = [p for p in todos if os.path.basename(p) not in ja]
    print(f"\nRodada {rodada}: já enviados {len(ja)} | faltam {len(faltam)}")
    if not faltam:
        break
    for i in range(0, len(faltam), 500):
        lote = faltam[i:i + 500]
        try:
            tab.add_files(files=lote, max_parallelization=8)
            print(f"  lote {i // 500 + 1}: ok ({i + len(lote)}/{len(faltam)})")
        except Exception as e:
            print(f"  lote {i // 500 + 1}: falhou ({type(e).__name__}); recomeça na próxima rodada")
            time.sleep(30)
            break

print("\nTotal no dataset:", len(ja_enviados()), "de", len(todos))

In [ ]:
# Bloco R4 — marcar as imagens do teste do Kaggle e comparar com o treino
ds = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018", version="next")
t_sub = None
for t in ds.list_tables():
    t.get()
    if t.properties.get("name", "").startswith("stage_2_sample_submission"):
        t_sub = t
sub = t_sub.to_pandas_dataframe()
print("Colunas:", sub.columns.tolist(), "| linhas:", len(sub))

mapa["kaggle_teste"] = mapa["subset_img_id"].isin(set(sub["patientId"]))
print("IDs do teste encontrados no mapeamento:", mapa["kaggle_teste"].sum(), "de", sub["patientId"].nunique())

rot = mapa[mapa["classe_final"].isin(["Lung Opacity", "No Lung Opacity / Not Normal", "Normal"])]
print("\nClasse final × conjunto do Kaggle (False = treino, True = teste):")
print(pd.crosstab(rot["kaggle_teste"], rot["classe_final"], margins=True).to_string())
print("\nProporção de positivos:")
print(rot.groupby("kaggle_teste")["positivo"].mean().round(3).to_string())

p_teste = set(rot.loc[rot["kaggle_teste"], "nih_paciente"])
p_treino = set(rot.loc[~rot["kaggle_teste"], "nih_paciente"])
print(f"\nPacientes só no teste: {len(p_teste - p_treino)} | em treino e teste: {len(p_teste & p_treino)}")

mapa.to_csv("/out/rsna/rsna_mapa_rotulos.csv", index=False)
print("Salvo: /out/rsna/rsna_mapa_rotulos.csv")

In [ ]:
# Bloco R5 — subir a tabela de rótulos para o dataset
ds = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018:2t9m", version="next")
t = ds.table("rotulos")
if not t.exists():
    t.create(description="Rótulo final adjudicado (grupo Calculated), mapeamento RSNA→NIH, paciente NIH, Exclude e marca de teste do Kaggle")

with open("/out/rsna/rsna_mapa_rotulos.csv", "rb") as f:
    t.upload("rsna_mapa_rotulos.csv").create(f, type="delimited", wait_for_finish=True)

t.get()
print("Tabela 'rotulos':", t.properties.get("numRows"), "linhas")

In [ ]:
# Bloco R6 — máscara do pulmão no RSNA com o procedimento do Plus (inspeção de 100, às cegas)
import redivis
import os, glob, zipfile, time
import numpy as np, pandas as pd, pydicom, torch, cv2
import matplotlib.pyplot as plt
from lungs_segmentation.pre_trained_models import create_model

# 1. arquivos locais
todos = glob.glob("/scratch/rsna_dcm/**/*.dcm", recursive=True)
if len(todos) < 30000:
    with zipfile.ZipFile("/out/rsna/pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip") as z:
        z.extractall("/scratch/rsna_dcm")
    todos = glob.glob("/scratch/rsna_dcm/**/*.dcm", recursive=True)
caminho = {os.path.basename(p)[:-4]: p for p in todos}      # SOPInstanceUID -> arquivo
print("Arquivos locais:", len(todos))

# 2. treino do Kaggle com rótulo final + cabeçalho
mapa = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018:2t9m").table("rotulos").to_pandas_dataframe()
mapa["kaggle_teste"] = mapa["kaggle_teste"].astype(str).str.lower().eq("true")
mapa.to_csv("/out/rsna/rsna_mapa_rotulos.csv", index=False)   # restaura a cópia local completa
base = mapa[~mapa["kaggle_teste"] & mapa["classe_final"].isin(
    ["Lung Opacity", "No Lung Opacity / Not Normal", "Normal"])].copy()
base["caminho"] = base["SOPInstanceUID"].map(caminho)

def ler_cab(p):
    ds = pydicom.dcmread(p, stop_before_pixels=True)
    return (ds.get("ViewPosition"), ds.get("PatientAge"), ds.get("PatientSex"),
            ds.get("PhotometricInterpretation"), ds.get("BitsStored"), ds.get("Rows"), ds.get("Columns"))

t0 = time.time()
cols_cab = ["proj", "idade", "sexo", "fotometria", "bits", "linhas", "colunas"]
base[cols_cab] = pd.DataFrame([ler_cab(p) for p in base["caminho"]], index=base.index)
base["idade"] = pd.to_numeric(base["idade"], errors="coerce")
base.drop(columns=["caminho"]).to_csv("/out/rsna/rsna_treino_cabecalhos.csv", index=False)
print(f"Cabeçalhos lidos: {len(base)} em {(time.time() - t0) / 60:.1f} min")
print("Projeção:", base["proj"].value_counts().to_dict())
print("Fotometria:", base["fotometria"].value_counts().to_dict(), "| bits:", base["bits"].value_counts().to_dict())
print("Menores de 18:", int((base["idade"] < 18).sum()), "| sem idade:", int(base["idade"].isna().sum()))

# 3. amostra: 50 AP + 50 PA entre adultos, semente 2026
eleg = base[(base["idade"] >= 18) & base["proj"].isin(["AP", "PA"])]
insp = pd.concat([eleg[eleg["proj"] == "AP"].sample(50, random_state=2026),
                  eleg[eleg["proj"] == "PA"].sample(50, random_state=2026)]).reset_index(drop=True)
insp["n"] = range(1, 101)

# 4. procedimento congelado do Plus
model = create_model("resnet34").eval()

def para_8bits_pct(ds, p_lo=0.5, p_hi=99.5):
    px = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(px, [p_lo, p_hi])
    px = np.clip((px - lo) / max(hi - lo, 1e-6), 0, 1)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        px = 1.0 - px
    return (px * 255).round().astype(np.uint8)

def maior_componente(m):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    if n <= 1:
        return m.astype(np.uint8)
    return (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)

def segmentar_roi(img8, limiar=0.2):
    x = np.repeat(img8[..., None], 3, axis=2).astype(np.float32)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-6)
    x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)))[0].numpy()
    pulmoes = np.stack([maior_componente(m) for m in (p > limiar)])
    return x[..., 0], pulmoes.max(axis=0), pulmoes

res = [segmentar_roi(para_8bits_pct(pydicom.dcmread(caminho[s]))) for s in insp["SOPInstanceUID"]]
insp["area_roi"] = [r[1].mean() for r in res]
insp["area_p1"] = [r[2][0].mean() for r in res]
insp["area_p2"] = [r[2][1].mean() for r in res]
insp["razao_pulmoes"] = insp[["area_p1", "area_p2"]].min(axis=1) / insp[["area_p1", "area_p2"]].max(axis=1)
insp.to_csv("/out/rsna/roi_inspecao_100_rsna.csv", index=False)

# 5. folhas às cegas: só número e projeção
for pg in range(4):
    fig, axes = plt.subplots(5, 5, figsize=(20, 20))
    for k, ax in enumerate(axes.flat):
        i = pg * 25 + k
        ax.imshow(res[i][0], cmap="gray")
        ax.contour(res[i][1], levels=[0.5], colors="r", linewidths=1)
        ax.set_title(f"#{i + 1} {insp.loc[i, 'proj']}", fontsize=14)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"/out/rsna/roi_inspecao_rsna_{pg + 1}.png", dpi=60)
    plt.show()

print("Razão entre pulmões < 0,35 (alerta):", insp.loc[insp["razao_pulmoes"] < 0.35, "n"].tolist())

In [ ]:
# Bloco R7 — ampliar casos em dúvida (às cegas)
casos = [5, 11, 15, 17, 58, 70, 85, 91]
for parte, bloco in enumerate((casos[:4], casos[4:]), start=1):
    fig, axes = plt.subplots(len(bloco), 2, figsize=(14, 7 * len(bloco)))
    for (a1, a2), n in zip(axes, bloco):
        i = n - 1
        img = para_8bits_pct(pydicom.dcmread(caminho[insp.loc[i, "SOPInstanceUID"]]))
        roi_full = cv2.resize(res[i][1], (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        a1.imshow(img, cmap="gray")
        a1.set_title(f"#{n} {insp.loc[i, 'proj']} — sem máscara", fontsize=13)
        a1.axis("off")
        a2.imshow(img, cmap="gray")
        a2.contour(roi_full, levels=[0.5], colors="r", linewidths=1.5)
        a2.set_title(f"#{n} — área {insp.loc[i, 'area_roi']:.2f} · razão entre pulmões {insp.loc[i, 'razao_pulmoes']:.2f}",
                     fontsize=13)
        a2.axis("off")
    plt.tight_layout()
    plt.savefig(f"/out/rsna/roi_casos_ampliados_{parte}.png", dpi=60)
    plt.show()

In [ ]:
# Bloco R8b — #70 e #85, versão independente (não precisa rodar os blocos anteriores)
import os, glob
import numpy as np, pandas as pd, pydicom, torch, cv2, redivis
import matplotlib.pyplot as plt
from lungs_segmentation.pre_trained_models import create_model

insp = pd.read_csv("/out/rsna/roi_inspecao_100_rsna.csv")
model = create_model("resnet34").eval()

def para_8bits_pct(ds, p_lo=0.5, p_hi=99.5):
    px = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(px, [p_lo, p_hi])
    px = np.clip((px - lo) / max(hi - lo, 1e-6), 0, 1)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        px = 1.0 - px
    return (px * 255).round().astype(np.uint8)

def maior_componente(m):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    if n <= 1:
        return m.astype(np.uint8)
    return (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)

def segmentar_roi(img8, limiar=0.2):
    x = np.repeat(img8[..., None], 3, axis=2).astype(np.float32)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-6)
    x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)))[0].numpy()
    pulmoes = np.stack([maior_componente(m) for m in (p > limiar)])
    return x[..., 0], pulmoes.max(axis=0), pulmoes

tab_dcm = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018:2t9m").table("dicom:sjtd")
os.makedirs("/scratch/casos", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, n in zip(axes, [70, 85]):
    i = n - 1
    sop = insp.loc[i, "SOPInstanceUID"]
    print(f"#{n}: paciente NIH {insp.loc[i, 'nih_paciente']} | imagem {insp.loc[i, 'img_id']} | projeção {insp.loc[i, 'proj']}")
    local = glob.glob(f"/scratch/rsna_dcm/**/{sop}.dcm", recursive=True)
    if local:
        arq = local[0]
    else:
        tab_dcm.file(f"{sop}.dcm").download("/scratch/casos")
        arq = f"/scratch/casos/{sop}.dcm"
    img = para_8bits_pct(pydicom.dcmread(arq))
    _, roi, _ = segmentar_roi(img)
    roi_full = cv2.resize(roi, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    h = img.shape[0]
    ax.imshow(img[h // 2:], cmap="gray")
    ax.contour(roi_full[h // 2:], levels=[0.5], colors="r", linewidths=1.5)
    ax.set_title(f"#{n} — metade inferior", fontsize=14)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bloco R9 — fluxograma final do RSNA (treino do Kaggle) e duplicatas exatas
import pandas as pd, redivis

ds = redivis.user("ads_alinelima").dataset("rsna_pneumonia_2018:2t9m")
cab = pd.read_csv("/out/rsna/rsna_treino_cabecalhos.csv")
idx = ds.table("dicom:sjtd").to_pandas_dataframe()
idx["file_name"] = idx["file_name"].astype(str)
idx["SOPInstanceUID"] = idx["file_name"].str.split("/").str[-1].str.replace(".dcm", "", regex=False)
cab = cab.merge(idx[["SOPInstanceUID", "md5_hash"]], on="SOPInstanceUID", how="left")
cab["positivo"] = cab["classe_final"].eq("Lung Opacity")
cab["seguimento"] = cab["img_id"].str[9:12].astype(int)

print("Idades acima de 100 (possível erro de registro):", int((cab["idade"] > 100).sum()))

passos = []
x = cab;                                passos.append(("Treino do Kaggle com rótulo final", x))
x = x[x["idade"] >= 18];                passos.append(("Idade >= 18", x))
x = x[x["proj"].isin(["AP", "PA"])];    passos.append(("AP ou PA", x))

# duplicatas exatas (mesma regra do Plus)
dup = x[x.duplicated("md5_hash", keep=False)]
grp = dup.groupby("md5_hash")["nih_paciente"].nunique()
sai_entre = x["md5_hash"].isin(grp[grp > 1].index)
mesmo = x[x["md5_hash"].isin(grp[grp == 1].index)].sort_values(["md5_hash", "seguimento"])
sai_mesmo = x.index.isin(mesmo[mesmo.duplicated("md5_hash", keep="first")].index)
print(f"Duplicatas exatas: {len(dup)} arquivos em {dup['md5_hash'].nunique()} grupos "
      f"({int((grp > 1).sum())} grupos entre pacientes diferentes)")
x = x[~sai_entre & ~sai_mesmo];         passos.append(("Sem duplicata exata", x))
final = x.copy()

flux = pd.DataFrame([{"etapa": n, "imagens": len(t), "pacientes": t["nih_paciente"].nunique(),
                      "positivos": int(t["positivo"].sum())} for n, t in passos])
flux.to_csv("/out/rsna/fluxograma_rsna.csv", index=False)
print("\n" + flux.to_string(index=False))

pos = int(final["positivo"].sum())
print(f"\nPositivos esperados no teste (20%): ~{pos * 0.2:.0f} | meta 750")
print(f"Prevalência: {final['positivo'].mean():.1%}")

final["faixa_idade"] = pd.cut(final["idade"], [18, 40, 60, 75, 200], right=False,
                              labels=["18–39", "40–59", "60–74", "75+"])
for c in ["proj", "sexo", "faixa_idade"]:
    g = final.groupby(c, observed=True)["positivo"]
    print(f"\n=== {c}")
    print(pd.DataFrame({"imagens": g.size(), "prevalencia": g.mean().round(3)}).to_string())

final.to_csv("/out/rsna/rsna_elegiveis.csv", index=False)
print("\nSalvo: /out/rsna/rsna_elegiveis.csv |", len(final), "linhas")